# World models: предсказывать переходы

Action-conditioned динамика, rollout и накопление ошибки.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

World model учится предсказывать следующее наблюдение или скрытое состояние с учётом действия. При обучении она часто видит настоящие предыдущие состояния; при rollout получает собственные предсказания, поэтому ошибка накапливается.

Ниже контролируемая линейная среда и небольшая нейросеть. Это учебная динамика, не DIAMOND или Genie. В World Models Studio собственные модели имеют путь обучения; у foundation-моделей смотрите отдельные флаги inference/training. Доступные веса не означают наличие fine-tuning. Не делите переходы одного эпизода случайно между train/validation.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
states = torch.randn(128, 2); actions = torch.randn(128, 1)
next_states = 0.9 * states + actions * torch.tensor([[0.2, -0.1]])
model = nn.Sequential(nn.Linear(3, 24), nn.Tanh(), nn.Linear(24, 2))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
for step in range(100):
    prediction = model(torch.cat([states, actions], -1))
    loss = F.mse_loss(prediction, next_states)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
real = torch.tensor([[1., -1.]]); imagined = real.clone(); errors = []
with torch.no_grad():
    for step in range(20):
        action = torch.tensor([[0.3]])
        real = 0.9 * real + action * torch.tensor([[0.2, -0.1]])
        imagined = model(torch.cat([imagined, action], -1))
        errors.append(F.mse_loss(imagined, real).item())
print('one-step train MSE:', loss.item())
plt.plot(errors, color='#d73535'); plt.xlabel('rollout step'); plt.ylabel('rollout MSE'); plt.show()

## Попробуйте сами

Увеличьте rollout до 100 шагов и задайте действия вне диапазона train. Низкий one-step loss гарантирует ли устойчивую симуляцию?

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [DIAMOND](https://arxiv.org/abs/2405.12399)
- [DreamerV3](https://arxiv.org/abs/2301.04104)